The TIMS data used for these indicators are downloaded here at this tool:  https://tims.berkeley.edu/tools/safetypm

Choose the "SWITRS" option for the *Facility data source*.

The raw data needs to be downloaded at every level within the SACOG region and stored here:  https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FSafe%20Equitable%20Resilient%20Infrastructure%2FSafety%2FTIMS&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d


For example, in the "Fatalities" folder, there is a folder for counties, jurisdictions, MPO, and statewide.  Step through the tool and download the data one geography at a time.  For the "Counties" folder, us the *County* drop-down option and set it to "El Dorado".  Leave the *City* drop-down option set to "Show Countywide".  There should be two 6 charts total when you scroll down the webpage (2 for fatalities, 2 for serious injuries, and 2 for non-motorized), but for the "Fatalities" folder, only download a *csv* file from the first two charts.  Rename them to match the naming convention that is already in the "Counties" folder and copy/paste them to replace the old files.

Repeat this process for all counties and all jurisdictions.  Then pull the data for the SACOG MPO and CA statewide.

Then, repeat this process for the "Serious Injuries" and "Non-Motorized" folders.


Once all the data is downloaded, rerun this script (make sure that you renamed the files properly to import correctly here and run smoothly).  Check the plots to make sure that the data looks appropriate.  If the outputs look good, then uncomment all blocks of code that export the results.  Then rerun the script and check the outputs on SharePoint.

In [ ]:


# TIMS does not have an API available.  Data were manually downloaded from here https://tims.berkeley.edu/ and stored on SharePoint


import pandas as pd
import os
from pathlib import Path
import plotly.express as px
from IPython.display import display
import sys
sys.path.append(str(Path.cwd().parent.parent / 'config'))
import functions as func
import plot as pt

sys.path.append(str(Path.cwd().parent.parent / 'Data' / 'TIMS' / 'config'))
import tims


pd.set_option('display.max_columns', None)


# SharePoint OneDrive paths
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_OUT  = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data' / 'Safe Equitable Resilient Infrastructure' / 'Safety'
PATH_TIMS = PATH_OUT / 'TIMS'
FILE_ABOUT = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'



EXPORT=True


In [ ]:

year_start = 2012
year_end = 2025
years_to_import = range(year_start, year_end+1)

categories = [cat for cat in os.listdir(PATH_TIMS) if '.xlsx' not in cat]
categories.remove('Bicyclists vs Pedestrians')

print(categories, years_to_import)


In [ ]:


# Import data at the mpo level
df_tims3 = tims.mpo(categories, years_to_import)
display(df_tims3.head())

if EXPORT:
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data by MPO.xlsx', engine='xlsxwriter') as writer:
        df_tims3.to_excel(writer, index=False, sheet_name='All')


# Import data statewide
df_tims4 = tims.statewide(categories, years_to_import)
display(df_tims4.head())

if EXPORT:
    with pd.ExcelWriter(PATH_TIMS / 'TIMS SWITRS Data Statewide.xlsx', engine='xlsxwriter') as writer:
        df_tims4.to_excel(writer, index=False, sheet_name='Statewide')



In [ ]:

# Safety_1 Collision Rates
df_safety1_3 = df_tims3[['MPO', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                      , 'Fatalities_100 mvmt'     , 'Fatalities_100 mvmt_5 Year Rolling Average'
                                      , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                      , 'Serious injuries_100 mvmt', 'Serious injuries_100 mvmt_5 Year Rolling Average']]
df_safety1_4 = df_tims4[['State', 'Year', 'Fatalities'              , 'Fatalities_5 Year Rolling Average'
                                        , 'Fatalities_100 mvmt'     , 'Fatalities_100 mvmt_5 Year Rolling Average'
                                        , 'Serious injuries'        , 'Serious injuries_5 Year Rolling Average'
                                        , 'Serious injuries_100 mvmt', 'Serious injuries_100 mvmt_5 Year Rolling Average']]

# Safety_2 Non-Motorized
df_safety2_3 = df_tims3[['MPO', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                      , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]
df_safety2_4 = df_tims4[['State', 'Year', 'Non motorized fatalities', 'Non motorized fatalities_5 Year Rolling Average'
                                        , 'Non motorized serious in', 'Non motorized serious in_5 Year Rolling Average']]



if EXPORT:

    # Update overall about documentations workbook
    sample_type = 'TIMS'
    indicators = ['Safety_1', 'Safety_2']
    params = {
        'sample': sample_type
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
    }

    categories = ['Collision Rates', 'Non-Motorized']
    paths = [PATH_OUT, PATH_SERVER]

    for path in paths:
        for category in categories:
            if category == 'Collision Rates':
                params['indicator'] = 'Safety_1'
                dt_geo = {
                    'MPO': df_safety1_3,
                    'Statewide': df_safety1_4
                    }
            if category == 'Non-Motorized':
                params['indicator'] = 'Safety_2'
                dt_geo = {
                    'MPO': df_safety2_3,
                    'Statewide': df_safety2_4
                    }
            for geography, df in dt_geo.items():
                params['geo']=geography
                df_about = func.write_about(params)
                if path == PATH_OUT:
                    file_out = path / f'{params['indicator']} {category}' / f'{params['indicator']} {params['geo']} {params['sample']}.xlsx'
                if path == PATH_SERVER:
                    file_out = path / f'{params['indicator']} {params['geo']} {params['sample']}.xlsx'
                print('Excel files exported here: ' + str(file_out))
                with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
                    df_about.to_excel(writer, index=False, sheet_name='About'  , header=False)
                    df      .to_excel(writer, index=False, sheet_name=geography              )

                
